# Lesson 00 — Setup, architecture and the corpus

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/00-setup/colab.ipynb)

*The environment, the corpus, and the one architectural decision everything else rests on.*

**What you will learn**

- What Retrieval-Augmented Generation (RAG) is, and the five stages every RAG system has
- How this course is organised: thin lessons over one shared library, `ragkit/`
- Why the code is written against three *protocols* (Embedder, Reranker, Generator) and never against a vendor SDK
- What an embedding is, and why queries and documents are embedded differently
- What is in the corpus you will search for the rest of the course

| | |
|---|---|
| ⏱️ Time | 30–40 min |
| 🖥️ Runtime | CPU is fine |
| 🤖 Needs the generator (Ollama) | yes — for some cells |


> **How to use this notebook.** Run the cells top to bottom (`Shift+Enter`). Every
> time you see **🔮 Predict**, stop and write down what you expect *before* running
> the next cell — the gap between prediction and result is where the learning is.
> Answers to the check-your-understanding questions are hidden behind ▶ toggles.

## 0. Setup

This cell clones the course repo into the Colab runtime, installs the
dependencies, and (if `USE_LLM` is ticked) installs and starts
[Ollama](https://ollama.com) with the small `qwen2.5:1.5b` generator. It is safe
to re-run. Everything in the repo's `ragkit/` library becomes importable.

If an import fails after the install, use **Runtime → Restart session** and run
this cell again.

In [ ]:
# @title ⚙️ Setup — run me first
USE_LLM = True  # @param {type:"boolean"}
BRANCH = "main"  # @param {type:"string"}

import subprocess
import sys
from pathlib import Path


def _find_repo():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "ragkit").is_dir() and (p / "colab" / "colab_setup.py").exists():
            return p
    return None


REPO = _find_repo()
if REPO is None:
    REPO = Path("/content/rag-from-scratch-to-production")
    if not REPO.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                BRANCH,
                "https://github.com/safar-byte/rag-from-scratch-to-production.git",
                str(REPO),
            ],
            check=True,
        )
sys.path.insert(0, str(REPO / "colab"))
from colab_setup import compare_runs, eval_frames, results_frame, setup  # noqa: F401

LLM_READY = setup(llm=USE_LLM)

## 1. What is RAG?

A large language model (LLM) only knows what was in its training data. Ask it about
*your* documents — an internal runbook, last week's incident report — and it either says
it does not know or, worse, invents a plausible answer.

**Retrieval-Augmented Generation** fixes that by doing a search first and handing the
model what it found:

```
                   ┌───────────────────────── offline, once ─────────────────────────┐
 documents ──► load ──► chunk ──► embed ──► store (vector index + keyword index)
                   └─────────────────────────────────────────────────────────────────┘

                   ┌───────────────────────── online, per question ──────────────────┐
 question ──► embed ──► search the store ──► top-k chunks ──► prompt ──► LLM ──► answer
                   └─────────────────────────────────────────────────────────────────┘
```

That is genuinely all RAG is. Every technique in lessons 04–11 modifies the *search*
step, and lesson 12 is about running it reliably.

Two facts about RAG drive the whole course:

1. **Almost every bad RAG answer is a bad retrieval wearing a fluent voice.** If the
   right passage never reaches the prompt, no model can answer correctly. So most of
   the course is about retrieval, not about the LLM.
2. **You cannot improve what you cannot measure.** Lesson 03 builds an evaluation
   harness *before* any optimisation, and every technique after it must earn its
   place with a number in `benchmarks/results.md`. Several techniques in this course
   *lost* — and those results are kept, because a negative result is a finding.

## 2. How the repository is organised

The lessons are thin. The real code accumulates in one library, `ragkit/`, so by
lesson 12 the lessons *are* a production service rather than thirteen disconnected
scripts.

| Folder | What lives there |
|---|---|
| `ragkit/config.py` | Settings and profile selection — the only place model names live |
| `ragkit/types.py` | `Document`, `Chunk`, `Scored`, `Citation`, `Usage`, `Answer` |
| `ragkit/providers/` | The three protocols, plus local (offline) and cloud backends |
| `ragkit/ingest/` | Loading and chunking documents |
| `ragkit/store/` | The vector store (Chroma, embedded — no server needed) |
| `ragkit/retrieve/` | Dense, BM25, hybrid, reranking, parent-document, routing |
| `ragkit/query/` | Query transformations (rewrite, HyDE, multi-query, step-back) |
| `ragkit/generate/` | Prompt assembly, answer generation, refusal gate |
| `ragkit/eval/` | Golden question set, metrics, judges, the eval runner |
| `ragkit/agent/`, `ragkit/graph/` | Agentic RAG and GraphRAG |
| `ragkit/api/` + `ui/` | The HTTP service and the retrieval inspector |
| `data/` | The corpus: 16 short documents about retrieval systems |
| `benchmarks/results.md` | Every measured result, including the ones that lost |

Let's look at the tree.

In [ ]:
!find ragkit -name "*.py" | sort | head -60

## 3. Check the environment

Lesson 00's `run.py` probes everything that produces confusing errors later. We can run
the lesson script directly from the notebook with `!python`.

> On Colab the `python` row may say *missing* because Colab's Python is not 3.12.
> That is fine here: the 3.12 pin exists because torch had no wheels for 3.14 on the
> author's machine, and Colab ships torch preinstalled. What matters is that `core`,
> `local models` and `corpus` pass. `index` will say *missing* until we ingest in
> lesson 01.

In [ ]:
!{sys.executable} lessons/00-setup/run.py

## 4. The one decision that matters: three protocols

Open `ragkit/providers/base.py`. It defines three **protocols** — `Embedder`,
`Reranker`, `Generator` — and nothing else in the library imports a vendor SDK
(`torch`, `sentence_transformers`, `anthropic`, `voyageai`).

In [ ]:
print(Path("ragkit/providers/base.py").read_text())

### What is a `Protocol`?

A `typing.Protocol` is *structural* typing — "duck typing that a type checker
understands". A class satisfies `Embedder` if it **has the right methods**; it does
not need to inherit from anything. With `@runtime_checkable` you can even check it with
`isinstance`.

Let's prove it by writing the world's worst embedder — a hash of the characters — and
checking that it counts as an `Embedder`. The test suite does exactly this
(`tests/fakes.py`): fakes that are semantically meaningless but completely sufficient
to prove the plumbing works.

In [ ]:
import hashlib

from ragkit.providers.base import Embedder, Generator


class HashEmbedder:
    """Meaningless vectors, valid shape. Enough to test plumbing."""

    dimension = 8
    name = "hash"

    def _vec(self, text: str) -> list[float]:
        digest = hashlib.sha256(text.encode()).digest()
        return [b / 255 for b in digest[: self.dimension]]

    def embed_documents(self, texts):
        return [self._vec(t) for t in texts]

    def embed_query(self, text):
        return self._vec(text)


print("HashEmbedder is an Embedder? ", isinstance(HashEmbedder(), Embedder))
print("HashEmbedder is a Generator?", isinstance(HashEmbedder(), Generator))

### Why bother with this abstraction?

1. **You can iterate for free.** Most of the time spent on a RAG system goes into
   retrieval logic: chunk boundaries, fusion, shortlist depth. None of that needs a
   frontier model. Running experiments locally at zero cost means you run more of
   them — and running more of them is the whole game.
2. **You can test without mocking the world.** A test suite that needs a GPU is a
   test suite nobody runs.
3. **Anyone can clone and run it.** A repo that demands two API keys before it
   prints anything is a repo most people close.

The signal you got the boundary wrong: a lesson file importing `anthropic` or `torch`.
Let's check that nothing outside `providers/` does:

In [ ]:
!grep -rn "import anthropic\|import voyageai\|import torch\|sentence_transformers" ragkit/ | grep -v providers/ || echo "✓ no vendor SDK imported outside ragkit/providers/"

## 5. Profiles: local (default) and cloud

`RAG_PROFILE` selects the backend. Everything else is identical.

| | `local` (default) | `cloud` (opt-in) |
|---|---|---|
| Embeddings | BGE-small (`BAAI/bge-small-en-v1.5`), 384-dim | Voyage |
| Reranking | `cross-encoder/ms-marco-MiniLM-L-6-v2` | Voyage rerank |
| Generation | Ollama `qwen2.5:1.5b` | Claude |
| Keys | none | `ANTHROPIC_API_KEY`, `VOYAGE_API_KEY` |

The cloud profile is **two vendors**, because Anthropic serves no embeddings endpoint.
That is the honest shape of a production RAG stack and the reason there are three
protocols rather than one "LLM" object.

All settings live in one `pydantic-settings` class, read from environment variables
and an optional `.env` file:

In [ ]:
from ragkit.config import Settings, get_settings

settings = get_settings()
for field in [
    "profile",
    "local_embedding_model",
    "local_rerank_model",
    "local_generation_model",
    "vector_store",
    "chroma_path",
    "chunk_size",
    "chunk_overlap",
    "top_k",
    "rerank_candidates",
]:
    print(f"{field:24s} {getattr(settings, field)}")

Settings can be overridden per-instance — handy in a notebook. `get_settings()` is
cached, so for experiments we construct a fresh `Settings(...)` instead of mutating the
shared one:

In [ ]:
experiment = Settings(chunk_size=256, chunk_overlap=32)
print(experiment.chunk_size, experiment.chunk_overlap, "| shared:", get_settings().chunk_size)

In [ ]:
!{sys.executable} -m pytest tests/test_config.py -q

**Question: `test_default_profile_is_local` asserts a *default*. Tests usually assert behaviour — why is this one worth writing?**

<details><summary>▶ Show answer</summary>

Because the default *is* the behaviour a newcomer experiences. If it ever flipped to
`cloud`, a fresh clone would crash asking for API keys before printing anything —
the repo's central promise ("runs offline with no keys") would break silently for
exactly the people least able to diagnose it.

</details>

## 6. Embeddings, by hand

An **embedding model** turns text into a vector (a list of numbers) such that texts
with similar *meaning* end up close together. "Close" is measured with **cosine
similarity** — the cosine of the angle between two vectors:

$$\text{cos}(a, b) = \frac{a \cdot b}{\lVert a \rVert\,\lVert b \rVert}$$

It ranges from −1 to 1; higher means more similar. If vectors are normalised to length
1 (BGE's are), cosine is just the dot product.

Let's load the local embedder through the provider factory — the same call every
lesson uses — and embed a few sentences. (First run downloads ~130MB.)

In [ ]:
import numpy as np

from ragkit.providers import get_embedder

embedder = get_embedder()
print(embedder.name, "→", embedder.dimension, "dimensions")

sentences = [
    "The invoice was charged twice to my card.",
    "Duplicate transactions appear on the statement.",
    "Chunk overlap repeats text between neighbouring chunks.",
    "TX-4491",
]
vectors = np.array(embedder.embed_documents(sentences))
print("vector shape:", vectors.shape, "| first 6 numbers:", vectors[0][:6].round(3))
print("norms:", np.linalg.norm(vectors, axis=1).round(3), "← already normalised")

In [ ]:
import pandas as pd

sim = vectors @ vectors.T  # dot product == cosine for unit vectors
labels = [s[:28] for s in sentences]
pd.DataFrame(sim, index=labels, columns=labels).round(3)

🔍 **Read the matrix.** "charged twice" and "duplicate transactions" share *no words*
but score high — that is what embeddings are for (bridging **vocabulary mismatch**).
`TX-4491` is close to nothing in particular: an identifier carries almost no meaning,
which is exactly where embeddings struggle and keyword search (lesson 04) shines.

### The asymmetry: queries and documents are embedded differently

BGE is trained with an **instruction prefix on the query side only**:
`"Represent this sentence for searching relevant passages: "`. That is why the
`Embedder` protocol has two methods, `embed_query` and `embed_documents`.

🔮 **Predict:** if you embed a *query* without the prefix, will its similarity to the
right passage go up or down? And will retrieval get *better* or *worse*?

In [ ]:
passage = (
    "Exit status 75 means the indexer could not acquire the lock on the index "
    "directory because another indexer process is still running."
)
query = "what does exit status 75 mean"

doc_vec = np.array(embedder.embed_documents([passage])[0])
with_prefix = np.array(embedder.embed_query(query))  # what ragkit does
without_prefix = np.array(embedder.embed_documents([query])[0])  # no instruction

print(f"query WITH instruction prefix   : {with_prefix @ doc_vec:.4f}")
print(f"query WITHOUT instruction prefix: {without_prefix @ doc_vec:.4f}")
print("prefix used:", repr(embedder.QUERY_INSTRUCTION))

The absolute score moved — but **absolute cosine values are not comparable across two
ways of embedding**. The prefix shifts *every* query vector, so what matters is whether
the right passage still *ranks* first among all the others. One pair cannot tell you
that. Let's test it properly: split the corpus into paragraphs, and for each of the
34 answerable test questions (from lesson 03's golden set) find the rank of the first
paragraph from a correct document — with and without the prefix.

In [ ]:
from ragkit.eval.golden import load_golden
from ragkit.ingest import load_documents

paragraphs = [(d.doc_id, p) for d in load_documents() for p in d.text.split("\n\n") if len(p) > 80]
P = np.array(embedder.embed_documents([p for _, p in paragraphs]))
questions = [q for q in load_golden() if q.relevant]


def rank_of_answer(query_vec, relevant):
    order = np.argsort(-(P @ query_vec))
    return next(r for r, i in enumerate(order, 1) if paragraphs[i][0] in relevant)


ranks = pd.DataFrame(
    {
        "with prefix": [
            rank_of_answer(np.array(embedder.embed_query(q.question)), q.relevant)
            for q in questions
        ],
        "without prefix": [
            rank_of_answer(np.array(embedder.embed_documents([q.question])[0]), q.relevant)
            for q in questions
        ],
    }
)
print(f"{len(paragraphs)} paragraphs, {len(questions)} questions\n")
pd.DataFrame(
    {
        "answer ranked 1st": (ranks == 1).mean(),
        "answer in top 3": (ranks <= 3).mean(),
        "mean rank": ranks.mean(),
    }
).round(3)

**Look at the result before reading on.** When this notebook was written, the
prefix made *no improvement* on this corpus — the no-prefix version was marginally
ahead, by a single question. The BGE v1.5 model card itself says the instruction is
optional for short-query retrieval.

That is a better lesson than the one the code comment teaches. `ragkit` applies the
prefix because that is how BGE was trained, and on a larger corpus it may help — but
**"the model was trained that way" is a reason to try something, not a measurement**.
Two things to carry forward:

1. **Nothing raises.** Whichever way you embed, the system keeps working; only a
   measurement over many questions tells you which is better.
2. **One example proves nothing.** The single pair above moved; the ranking over 34
   questions barely did. Lesson 03 builds the harness that makes this kind of check
   routine.

### The data types that flow through the pipeline

`ragkit/types.py` is short and worth reading in full. Note that `Chunk` has both
`text` and `context`, and that `Scored` keeps a *dictionary* of per-stage scores
rather than a single float.

In [ ]:
print(Path("ragkit/types.py").read_text())

**Question: `Chunk` has `text` and `context`, joined by `embed_text`. Nothing sets `context` yet. Why keep `text` pristine rather than concatenating the two at ingest?**

<details><summary>▶ Show answer</summary>

Lesson 07 fills `context` with an LLM-written sentence situating the chunk in its
document. That *generated* text helps search, so it is embedded. But citations must
point at *source* text — if you cite generated text, the system can no longer prove
anything it says. Keeping the fields separate lets you embed `context + text` while
showing and citing only `text`.

</details>

## 7. The corpus

`data/` holds 16 short **original** documents about retrieval systems. They were
written for this repo rather than scraped: the licence is unambiguous and the facts are
specific enough to write verifiable test questions against. It is also deliberately
awkward in places — some facts are split across documents, some terms are defined in
one file and used in another — because a corpus that is too clean makes every
retrieval strategy look equally good.

In [ ]:
from ragkit.ingest import load_documents

documents = load_documents()
pd.DataFrame(
    [
        {"doc_id": d.doc_id, "title": d.title, "chars": len(d.text), "words": len(d.text.split())}
        for d in documents
    ]
)

In [ ]:
print(Path("data/README.md").read_text())

In [ ]:
from IPython.display import Markdown, display

display(Markdown(Path("data/07-operations.md").read_text()[:2500] + "\n\n*…truncated…*"))

🔍 `07-operations.md` is full of **identifiers** — error codes, ports, config keys like
`FUSION_CONSTANT`. Those are what keyword search is good at and embeddings are bad at.
Keep this file in mind; it comes back in lessons 01, 04 and 09.

## 8. The generator

If you ticked `USE_LLM`, Ollama is running in the background with `qwen2.5:1.5b` — a
small, *non-reasoning* instruct model. It was chosen by measurement: on the author's
CPU-only machine a 4B reasoning model took **over 10 minutes** for one RAG query because
it "thinks" for ~1000 tokens first. The generator only needs to read the context it is
handed; retrieval quality is what this course teaches.

Let's talk to it through the `Generator` protocol.

In [ ]:
import time

if LLM_READY:
    from ragkit.providers import get_generator

    generator = get_generator()
    started = time.perf_counter()
    text, usage = generator.generate(
        system="You are a concise assistant.",
        prompt="In one sentence, what is retrieval-augmented generation?",
        max_tokens=80,
    )
    print(text)
    print(
        f"\n{usage.input_tokens} in / {usage.output_tokens} out · "
        f"${usage.cost_usd} · {time.perf_counter() - started:.1f}s"
    )
else:
    print("Generator not available — tick USE_LLM in the setup cell to try this.")

Note `cost_usd = 0.0`: local inference is free, and the cost column stays honest at
zero. On the cloud profile the same `Usage` object carries real token prices.

## 9. Run the whole test suite

120 tests, all offline, no models — because the fakes stand in for the providers.
Quality is measured by the eval harness (lesson 03), not by unit tests.

In [ ]:
!{sys.executable} -m pytest -q 2>&1 | tail -5

## 10. Check your understanding

**1. What are the five stages of a RAG pipeline, and which of them happen at query time?**

<details><summary>▶ Show answer</summary>

Load → chunk → embed → store happen once, offline (ingest). At query time: embed the
question → search → put the top-k chunks in a prompt → generate.

</details>

**2. In `ragkit/providers/local.py`, `QUERY_INSTRUCTION` is applied in `embed_query` but not `embed_documents`. What breaks if you apply it to both? To neither? How would you notice?**

<details><summary>▶ Show answer</summary>

Both: every document vector is pushed towards "query-shaped" text, which BGE was
not trained for, so query–document similarity becomes less discriminative.
Neither: queries lose the instruction BGE was trained with — which, as section 6
measured, cost nothing on *this* corpus. **In every case nothing raises.** The only
way to notice either mistake, or to discover it was not a mistake at all, is an
evaluation over many questions (lesson 03).

</details>

**3. If you wrote both the corpus *and* the questions that test retrieval over it, what result would you expect — and should you trust it?**

<details><summary>▶ Show answer</summary>

You would expect suspiciously high scores, because questions written from the
documents reuse the documents' vocabulary, which every retriever matches easily.
You should not trust it. Lesson 03 finds exactly this: many questions are
"saturated" at a perfect score, so they cannot distinguish one technique from
another.

</details>

## ✍️ Exercise

1. Find one place where the corpus is "deliberately awkward" — a term defined in one
   document and used in another. (Hint: grep for a config key across `data/`.)
2. Add a fourth sentence to the embedding similarity matrix in section 6 that you
   predict will be close to "charged twice". Were you right?

In [ ]:
!grep -l "FUSION_CONSTANT\|RETRIEVAL_FANOUT" data/*.md

## Key takeaways

1. RAG = search first, then generate from what was found. Retrieval quality decides answer quality.
2. `ragkit` is written against three protocols, so the same code runs offline (local models, free) or in the cloud (Claude + Voyage).
3. Embeddings bridge vocabulary mismatch but carry little signal for identifiers.
4. Queries and documents are embedded through different methods; mixing them up fails silently.
5. The corpus is small and was written alongside its test questions — remember to be suspicious of results on it.

## Next

**[Lesson 01 — Naive RAG, end to end](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/01-naive-rag/colab.ipynb)**